<div style="width:100%; box-sizing:border-box; border-radius:14px; background:#141110; padding:34px 28px 30px 28px;">
<p style="margin:0; font-family:'Courier New', monospace; font-size:12px; font-weight:bold; letter-spacing:4px; color:#E8C97A;">BIOHUB | CELL TRACKING DURING DEVELOPMENT</p>
<p style="margin:12px 0 0 0; font-family:Georgia, serif; font-size:37px; font-weight:700; line-height:1.15; color:#ffffff;">A dividing nucleus gets<br>smaller, not dimmer</p>
<p style="margin:14px 0 0 0; font-family:'Courier New', monospace; font-size:12px; letter-spacing:2px; color:#E8C97A;">ONE ANIMATION &nbsp;|&nbsp; ONE MEASUREMENT &nbsp;|&nbsp; NO EXTRA PACKAGES</p>
</div>

I wanted to watch a division rather than read about one. So this animates a labelled division from the training data, and then measures the thing the animation appears to show.

The measurement is the point of the notebook, because my first version of it was wrong and the fix is reusable.

Divisions are rare here: 151 labelled ones across 199 training films, and the metric weights them at 0.1. It is worth knowing what one looks like.

Nothing below needs a package the Kaggle image lacks. `zarr` is not installed, so the `.geff` label arrays are read straight from their zstd chunks with `zstandard`, and the image chunks with `blosc2`.

## Chapter 1 &middot; Reading the data without zarr

In [ ]:
from pathlib import Path
from collections import defaultdict
import json, io, base64
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import blosc2, zstandard
from PIL import Image
from IPython.display import HTML, display

TRAIN = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
_ZDEC = zstandard.ZstdDecompressor()

def read_label_array(path):
    """A .geff array is a zstd-compressed zarr v3 chunk described by its own zarr.json.

    The Kaggle image has zstandard but not zarr, so this reads the chunk files directly
    rather than pulling in a package that is not installed.
    """
    meta = json.loads((path / 'zarr.json').read_text())
    dtype = np.dtype(meta['data_type']).newbyteorder('<')
    shape = tuple(meta['shape'])
    chunks = tuple(meta['chunk_grid']['configuration']['chunk_shape'])
    out = np.full(shape, meta.get('fill_value', 0), dtype=dtype)
    if 0 in shape:
        return out
    want = int(np.prod(chunks)) * dtype.itemsize
    grid = tuple(int(np.ceil(s / c)) for s, c in zip(shape, chunks))
    for idx in np.ndindex(*grid):
        blob = path / ('c' + ''.join('/' + str(i) for i in idx))
        if not blob.exists():
            continue
        block = np.frombuffer(_ZDEC.decompress(blob.read_bytes(), max_output_size=want),
                              dtype=dtype).reshape(chunks)
        region = tuple(slice(i * c, min((i + 1) * c, s)) for i, c, s in zip(idx, chunks, shape))
        out[region] = block[tuple(slice(0, r.stop - r.start) for r in region)]
    return out

def read_volume(stem, t):
    """One timepoint of the image: (64, 256, 256) uint16. Blosc container, zstd inside."""
    raw = (TRAIN / f'{stem}.zarr' / '0' / 'c' / str(t) / '0' / '0' / '0').read_bytes()
    return np.frombuffer(blosc2.decompress2(raw), dtype='<u2').reshape(64, 256, 256)

def read_graph(stem):
    g = TRAIN / f'{stem}.geff'
    ids = read_label_array(g / 'nodes/ids')
    cols = [read_label_array(g / f'nodes/props/{a}/values') for a in 'tzyx']
    edges = read_label_array(g / 'edges/ids')
    pos = {int(i): tuple(int(c[k]) for c in cols) for k, i in enumerate(ids)}
    kids, parent = defaultdict(list), {}
    for s, d in edges:
        kids[int(s)].append(int(d)); parent[int(d)] = int(s)
    return pos, kids, parent

VOXEL_UM = (1.625, 0.40625, 0.40625)   # z, y, x
stems = sorted(p.stem for p in TRAIN.glob('*.geff'))
print(len(stems), 'films')

## Chapter 2 &middot; Choosing a division a single projection can show

A division whose daughters drift apart in z cannot be told in one plane: any crop has to pick a z band, and one daughter falls out of it. I hit exactly that on my first attempt, so the filter below asks for daughters that stay within a few slices of each other and separate clearly in the projection plane.

That is a choice about legibility, not a typical event.

In [ ]:
def ancestors(n, parent, k):
    out = [n]
    for _ in range(k):
        n = parent.get(n)
        if n is None:
            break
        out.append(n)
    return out[::-1]

def descendants(n, kids, k):
    out = [n]
    for _ in range(k):
        c = kids.get(n, [])
        if len(c) != 1:
            break
        n = c[0]; out.append(n)
    return out

BEFORE, AFTER = 6, 10

def candidates(stem):
    """Divisions a single projection can actually show.

    A division whose daughters drift apart in z cannot be told in one plane: the crop has to
    pick a z band and one daughter falls out of it. My first attempt hit exactly that. So this
    asks for daughters that stay within a few slices of each other, a shallow lineage overall,
    and clear separation in the projection plane.

    That is a choice about legibility, not a typical event, and it is worth saying so out loud.
    """
    pos, kids, parent = read_graph(stem)
    out = []
    for n, cs in kids.items():
        if len(cs) != 2:
            continue
        back = ancestors(n, parent, BEFORE)
        f1, f2 = descendants(cs[0], kids, AFTER), descendants(cs[1], kids, AFTER)
        if len(back) - 1 < 5 or len(f1) < 7 or len(f2) < 7:
            continue
        t, z, y, x = pos[n]
        if not (8 < t < 88 and 40 < y < 216 and 40 < x < 216 and 8 < z < 56):
            continue
        m = min(len(f1), len(f2))
        dz = max(abs(pos[f1[i]][1] - pos[f2[i]][1]) for i in range(m))
        sep = float(np.hypot(pos[f1[m-1]][2] - pos[f2[m-1]][2], pos[f1[m-1]][3] - pos[f2[m-1]][3]))
        span = max(pos[q][1] for q in back + f1 + f2) - min(pos[q][1] for q in back + f1 + f2)
        if dz <= 3 and sep >= 12 and span <= 8:
            out.append({'stem': stem, 'node': n, 't': t, 'z': z, 'y': y, 'x': x,
                        'max_dz': dz, 'z_span': span, 'separation_px': round(sep, 1)})
    return out

found = []
for stem in stems:
    found += candidates(stem)
    if len(found) >= 12:
        break
found.sort(key=lambda d: -d['separation_px'])
import pandas as pd
print(pd.DataFrame(found).head(8).to_string(index=False))

## Chapter 3 &middot; The animation

Yellow is the parent, blue and red the two daughters. The window follows the lineage, so the cell stays centred while the field drifts underneath it. The division frame is held longer.

In [ ]:
PICKED = found[0]
HALF_XY, HALF_Z = 34, 2
COLOUR = {'parent': '#f0d68a', 'daughter 1': '#7fb3d5', 'daughter 2': '#e08a7a'}

pos, kids, parent = read_graph(PICKED['stem'])
node = PICKED['node']
c1, c2 = kids[node]
lineage = {'parent': ancestors(node, parent, BEFORE),
           'daughter 1': descendants(c1, kids, AFTER),
           'daughter 2': descendants(c2, kids, AFTER)}

by_t = defaultdict(list)
for name, chain in lineage.items():
    for n in chain:
        by_t[pos[n][0]].append((name, pos[n]))
t0 = pos[node][0]
frames = sorted(by_t)

def crop(t):
    """Follow the lineage: the window centre is the mean of whatever nodes exist at t,
    so the cell stays put while the field drifts underneath it."""
    a = np.array([[p[1], p[2], p[3]] for _, p in by_t[t]], float).mean(0)
    cz, cy, cx = (int(round(v)) for v in a)
    vol = read_volume(PICKED['stem'], t)
    y1 = min(max(0, cy - HALF_XY), 256 - 2 * HALF_XY)
    x1 = min(max(0, cx - HALF_XY), 256 - 2 * HALF_XY)
    band = vol[max(0, cz - HALF_Z):min(64, cz + HALF_Z + 1),
               y1:y1 + 2 * HALF_XY, x1:x1 + 2 * HALF_XY]
    return band.max(0), y1, x1

shots = [crop(t) for t in frames]
pool = np.concatenate([s[0].ravel() for s in shots])
lo, hi = np.percentile(pool, 1), np.percentile(pool, 99.7)

def panel(k):
    img, oy, ox = shots[k]
    t = frames[k]
    fig, ax = plt.subplots(figsize=(3.2, 3.4), facecolor='#0d0d0d')
    ax.imshow(np.clip((img - lo) / (hi - lo), 0, 1), cmap='gray', vmin=0, vmax=1,
              interpolation='bilinear')
    for name, (_, _, py, px) in by_t[t]:
        ax.add_patch(Circle((px - ox, py - oy), 7.5, fill=False, lw=2.0, color=COLOUR[name]))
    ax.set_title(f't = {t}' + ('    DIVISION' if t == t0 else ''),
                 color='#f0d68a' if t == t0 else '#b8b8b8', fontsize=11)
    ax.axis('off')
    fig.tight_layout(pad=0.2)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='#0d0d0d')
    plt.close(fig)
    buf.seek(0)
    # ADAPTIVE on a smooth grey ramp posterises into visible contour bands, so quantise
    # with dithering instead: the banding is an artefact of the encoder, not the data.
    return Image.open(buf).convert('RGB').quantize(
        colors=255, method=Image.MEDIANCUT, dither=Image.FLOYDSTEINBERG)

panels = [panel(k) for k in range(len(frames))]
durations = [700 if frames[k] == t0 else 260 for k in range(len(frames))]
gif = io.BytesIO()
panels[0].save(gif, format='GIF', save_all=True, append_images=panels[1:],
               duration=durations, loop=0, optimize=True)
gif.seek(0)
Path('/kaggle/working/division.gif').write_bytes(gif.getvalue())
print(PICKED['stem'], 'node', node, '| division at t =', t0, '|', len(frames), 'frames')
display(HTML('<img src="data:image/gif;base64,%s" style="image-rendering:pixelated;width:420px">'
             % base64.b64encode(gif.getvalue()).decode()))

Watch the frames around the split. Just before it, at t = 22 to 24, the parent condenses into a short bright band. Just after it, the two daughters look fainter than the parent did. It is tempting to read that second part as the cell dimming.

That is where I went wrong, so it is worth measuring rather than believing.

## Chapter 4 &middot; What "fading" actually means

My first measurement was mean intensity in a fixed-radius ball at each annotated position. It showed dividing cells dropping to 0.84 of their starting value while ordinary cells stayed flat, which looked like a clean result.

It was an artefact of the ruler. A fixed-radius ball around a **smaller** object contains more background, so the mean falls even when nothing gets dimmer.

The fix is to measure three things in the same box instead of one:

| quantity | what it is | what it tracks |
|---|---|---|
| **peak** | brightest voxel above the local background | brightness, nearly independent of size |
| **volume** | voxels above halfway between background and peak | **size** |
| **mean** | the naive measure | a mixture of the two |

If volume falls while peak holds, the nucleus got smaller and did not get dimmer.

The control is paired: for each division, up to three ordinary single-child tracks **from the same film starting on the same frame**. Anything that drifts across a film, bleaching or focus, then hits both arms equally and cancels.

One more thing had to change before the numbers meant anything. Reading label graphs is cheap and reading image volumes is not, so my first run simply took the first 40 films. It found 5 divisions and every interval covered zero. The cell below surveys all 199 graphs first and spends the volume budget on the films that actually carry divisions.

In [ ]:
RADIUS, WINDOW = 5, 6

def probe(vol, z, y, x):
    """Three numbers from one small box, which is the whole point of this cell.

      peak   - brightest voxel above the local background. Barely depends on how big the
               object is, so it tracks BRIGHTNESS.
      volume - voxels at or above halfway between background and peak. This is SIZE.
      mean   - the naive measure. It mixes the two, and that is exactly the trap.
    """
    b = vol[max(0, z-RADIUS):z+RADIUS+1,
            max(0, y-RADIUS):y+RADIUS+1,
            max(0, x-RADIUS):x+RADIUS+1].astype(np.float32)
    if b.size == 0:
        return None
    peak, bg = float(b.max()), float(np.percentile(b, 10))
    if peak <= bg:
        return None
    return peak - bg, float((b >= bg + 0.5*(peak-bg)).sum()), float(b.mean())

def walk(kids, c, step):
    q = c
    for _ in range(step - 1):
        nxt = kids.get(q, [])
        if len(nxt) != 1:
            return None
        q = nxt[0]
    return q

def curve(stem, pos, kids, parent, n, is_division, cache):
    """Each track normalised to its OWN first frame, so films of different brightness pool."""
    t0 = pos[n][0]
    if not (WINDOW <= t0 < 100 - WINDOW):
        return None
    back = ancestors(n, parent, WINDOW)
    if len(back) != WINDOW + 1:
        return None
    P, V, M = [], [], []
    for m in back:
        t, z, y, x = pos[m]
        if t not in cache:
            cache[t] = read_volume(stem, t)
        r = probe(cache[t], z, y, x)
        if r is None:
            return None
        P.append(r[0]); V.append(r[1]); M.append(r[2])
    for step in range(1, WINDOW + 1):
        gp, gv, gm = [], [], []
        targets = kids[n] if is_division else kids.get(n, [])[:1]
        if not targets:
            return None
        for c in targets:
            q = walk(kids, c, step)
            if q is None:
                return None
            t, z, y, x = pos[q]
            if t not in cache:
                cache[t] = read_volume(stem, t)
            r = probe(cache[t], z, y, x)
            if r is None:
                return None
            gp.append(r[0]); gv.append(r[1]); gm.append(r[2])
        P.append(np.mean(gp)); V.append(np.mean(gv)); M.append(np.mean(gm))
    P, V, M = map(np.array, (P, V, M))
    if min(P[0], V[0], M[0]) <= 0:
        return None
    return P/P[0], V/V[0], M/M[0]

# Reading label graphs is cheap; reading image volumes is not. So survey every film's graph
# first and spend the volume budget only where there are divisions to measure. My first run
# took the first 40 films alphabetically, got 5 divisions, and every interval covered zero.
survey = []
for stem in stems:
    pos, kids, parent = read_graph(stem)
    divisions = [n for n, c in kids.items() if len(c) == 2
                 and WINDOW <= pos[n][0] < 100 - WINDOW]
    if divisions:
        survey.append((len(divisions), stem, pos, kids, parent, divisions))
survey.sort(key=lambda r: -r[0])
print(f'{len(survey)} of {len(stems)} films carry a usable division, '
      f'{sum(r[0] for r in survey)} divisions in total')

FILM_BUDGET = 60          # bounded on purpose, and the films are the division-richest ones
div_rows, ctl_rows = [], []
for k, (ndiv, stem, pos, kids, parent, divisions) in enumerate(survey[:FILM_BUDGET]):
    by_start = defaultdict(list)
    for n in pos:
        if len(kids.get(n, [])) == 1:
            by_start[pos[n][0]].append(n)
    cache = {}                      # one cache per FILM, not per division
    for n in divisions:
        r = curve(stem, pos, kids, parent, n, True, cache)
        if r is None:
            continue
        div_rows.append(r)
        # PAIRED control: same film, same start frame, up to three ordinary tracks.
        # Anything that drifts across the whole film (bleaching, focus) hits both arms equally.
        got = 0
        for m in by_start[pos[n][0]]:
            if got >= 3:
                break
            rc = curve(stem, pos, kids, parent, m, False, cache)
            if rc is not None:
                ctl_rows.append(rc); got += 1
    cache.clear()
    if (k + 1) % 10 == 0:
        print(f'  {k+1}/{min(FILM_BUDGET, len(survey))} films | '
              f'{len(div_rows)} divisions | {len(ctl_rows)} controls', flush=True)

stack = lambda rows, i: np.array([r[i] for r in rows])
DP, DV, DM = (stack(div_rows, i) for i in range(3))
CP, CV, CM = (stack(ctl_rows, i) for i in range(3))
lags = np.arange(-WINDOW, WINDOW + 1)
print(f'\n{len(DV)} divisions | {len(CV)} paired ordinary tracks | '
      f'{min(FILM_BUDGET, len(survey))} films read')

## Chapter 5 &middot; The result, with intervals

A difference without an interval is not a result, so every lag gets a bootstrap interval against the paired controls.

In [ ]:
rng = np.random.default_rng(0)

def boot(a, b, k, n=6000):
    """Median difference at one lag, with a bootstrap interval. No interval, no claim."""
    obs = np.median(a[:, k]) - np.median(b[:, k])
    s = np.array([np.median(rng.choice(a[:, k], len(a))) - np.median(rng.choice(b[:, k], len(b)))
                  for _ in range(n)])
    return obs, *np.percentile(s, [2.5, 97.5])

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1), sharex=True)
panels = [('volume  (size)', DV, CV, '#c0392b'),
          ('peak  (brightness)', DP, CP, '#2471a3'),
          ('mean  (the naive one)', DM, CM, '#7d6608')]
for ax, (title, A, B, col) in zip(axes, panels):
    for arr, lab, c in ((B, 'ordinary', '#7f8c8d'), (A, 'dividing', col)):
        med = np.median(arr, 0)
        ax.fill_between(lags, np.percentile(arr, 25, 0), np.percentile(arr, 75, 0),
                        color=c, alpha=0.15)
        ax.plot(lags, med, color=c, lw=2.2, marker='o', ms=3.5, label=lab)
    ax.axvline(0, color='#b8860b', lw=1.1, ls='--')
    ax.axhline(1.0, color='#bbbbbb', lw=0.8)
    ax.set_title(title); ax.set_xlabel('frames from the division'); ax.grid(alpha=0.28)
    ax.legend(fontsize=9)
axes[0].set_ylabel('relative to the track start')
fig.suptitle('A dividing nucleus gets smaller, not dimmer', y=1.02, fontsize=13)
fig.tight_layout()
plt.show()

for name, A, B in (('VOLUME', DV, CV), ('PEAK', DP, CP), ('MEAN', DM, CM)):
    print(f'\n{name}   dividing minus ordinary, 95% bootstrap interval')
    for k, dt in enumerate(lags):
        if dt < -2:
            continue
        o, lo_, hi_ = boot(A, B, k)
        flag = '  <-- clears zero' if (lo_ > 0 or hi_ < 0) else ''
        print(f'   dt={dt:+d}   {o:+6.3f}  [{lo_:+.3f}, {hi_:+.3f}]{flag}')

## Chapter 6 &middot; Reading it

The three panels disagree, and the disagreement is the finding.

**Volume** is down, and its intervals clear zero at seven of the nine lags shown: at -2 and -1 before the division (the -1 bound only just), then +1 through +5 after it. The deepest point is about `-0.27` at three frames after the division, and by +6 it is drifting back toward the controls.

**Peak** is not. Its only lag that clears zero is the division frame itself, where the nucleus is mid-split. At every lag after that the interval covers zero: the daughters are as bright as the parent was.

**Mean** falls and stays down from +1 through +6, which is what my first version measured and mistook for dimming. It was reading the volume drop through a fixed-size probe.

So a dividing nucleus is **smaller**, not dimmer, and the apparent fade in the animation is mostly geometry.

Smaller after the split is expected on its own: each daughter carries half of the parent's chromatin. Two parts of the result are not automatic.

**Brightness is kept.** Half the material in a smaller nucleus leaves the peak where it was, rather than halving it.

**The size change starts before the split.** Volume is already below the controls at -2 and -1. That is consistent with the chromatin condensing ahead of the division, which is what the bright band at t = 22 to 24 in the animation looks like, although a box measurement cannot separate condensation from other causes.

The effect is a window rather than a step: it peaks around +3 and is fading by +6. The frames that decide the division half of the metric, the split and the one after it, fall inside that window.

These numbers are relative to each track's own start. They say nothing against dividing nuclei being brighter than ordinary ones in absolute terms, which is a comparison between cells rather than over time.

Caveats I would want stated if I were reading this:

- The film budget is bounded and printed. It reads the division-richest films, so these divisions are not a uniform sample of all of them.
- Half-max volume in a small box is a crude size estimate on an anisotropic voxel grid (1.625 by 0.40625 by 0.40625 micrometres). It is good enough to separate size from brightness, which is all it is asked to do here, and not good enough to quote as a nuclear volume.
- The intervals are per-lag and uncorrected. Volume clearing zero at seven lags in a row is not something to read one lag at a time, but a single isolated lag elsewhere would not be worth much.

The animation is written to `/kaggle/working/division.gif` if you want to reuse it.
